#**SVM method**

#### Importing libraries

In [ ]:
!pip install biopython

In [ ]:
import statistics
import numpy as np
import pandas as pd
from Bio import BiopythonWarning
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils import ProtParamData
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier

#### Loading data

In [ ]:
data_file = "/content/SVM_data.tsv"
df = pd.read_csv(data_file, sep="\t",)
df = df[df["dataset_class"].isin(["Training"])]
df = df.reset_index(drop=True)
df

,protein_id,dataset_class,cv_subset,type,sequence
0,Q99MA2,Training,1,Positive,MAQAYWQCYPWLVLLCACAWSYPGPESLGREDVRDCSTNPPRLPVT...
1,P17948,Training,1,Positive,MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...
2,P41271,Training,1,Positive,MMLRVLVGAVLPAMLLAAPPPINKLALFPDKSAWCEAKNITQIVGH...
3,Q8I948,Training,1,Positive,MAFRMKLVVCIVLLSTLAVMSSADVYKGGGGGRYGGGRYGGGGGYG...
4,Q92154,Training,1,Positive,MELLVLTVLLMGTGCISAPWAAWMPPKMAALSGTCVQLPCRFDYPE...
...,...,...,...,...,...
8017,Q01981,Training,5,Negative,MPQPGSSVDFSNLLNPQNNTAIPAEVSNATASATMASGASLLPPMV...
8018,Q9H3N8,Training,5,Negative,MPDTNSTINLSLSTRVTLAFFMSLVAFAIMLGNALVILAFVVDKNL...
8019,P32349,Training,5,Negative,MDELLGEALSAENQTGESTVESEKLVTPEDVMTISSLEQRTLNPDL...
8020,G5EGE9,Training,5,Negative,MDVPSSSNVTGRRKRQVLDDDEDDGFRSTPLRKVRGTKKIRPADVV...


#### Adding signal peptide residue composition to the DataFrame

In [ ]:
amino_acids = ['A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']

for index, row in df.iterrows():
    seq = row["sequence"][:22]
    protein_analysis = ProteinAnalysis(seq)
    amino_count = protein_analysis.count_amino_acids()
    total = sum(amino_count.values())
    for aa in amino_acids:
        proportion = amino_count.get(aa, 0) / total if total > 0 else 0
        df.at[index, f"{aa}_composition"] = proportion

df

,protein_id,dataset_class,cv_subset,type,sequence,A_composition,C_composition,D_composition,E_composition,F_composition,...,M_composition,N_composition,P_composition,Q_composition,R_composition,S_composition,T_composition,V_composition,W_composition,Y_composition
0,Q99MA2,Training,1,Positive,MAQAYWQCYPWLVLLCACAWSYPGPESLGREDVRDCSTNPPRLPVT...,0.181818,0.136364,0.000000,0.000000,0.000000,...,0.045455,0.000000,0.045455,0.090909,0.000000,0.045455,0.000000,0.045455,0.136364,0.136364
1,P17948,Training,1,Positive,MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...,0.045455,0.090909,0.045455,0.000000,0.000000,...,0.045455,0.000000,0.000000,0.000000,0.000000,0.090909,0.090909,0.090909,0.045455,0.045455
2,P41271,Training,1,Positive,MMLRVLVGAVLPAMLLAAPPPINKLALFPDKSAWCEAKNITQIVGH...,0.181818,0.000000,0.000000,0.000000,0.000000,...,0.136364,0.000000,0.181818,0.000000,0.045455,0.000000,0.000000,0.136364,0.000000,0.000000
3,Q8I948,Training,1,Positive,MAFRMKLVVCIVLLSTLAVMSSADVYKGGGGGRYGGGRYGGGGGYG...,0.090909,0.045455,0.000000,0.000000,0.045455,...,0.136364,0.000000,0.000000,0.000000,0.045455,0.136364,0.045455,0.181818,0.000000,0.000000
4,Q92154,Training,1,Positive,MELLVLTVLLMGTGCISAPWAAWMPPKMAALSGTCVQLPCRFDYPE...,0.136364,0.045455,0.000000,0.045455,0.000000,...,0.090909,0.000000,0.045455,0.000000,0.000000,0.045455,0.090909,0.090909,0.045455,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8017,Q01981,Training,5,Negative,MPQPGSSVDFSNLLNPQNNTAIPAEVSNATASATMASGASLLPPMV...,0.045455,0.000000,0.045455,0.000000,0.045455,...,0.045455,0.181818,0.136364,0.090909,0.000000,0.136364,0.045455,0.045455,0.000000,0.000000
8018,Q9H3N8,Training,5,Negative,MPDTNSTINLSLSTRVTLAFFMSLVAFAIMLGNALVILAFVVDKNL...,0.045455,0.000000,0.045455,0.000000,0.090909,...,0.090909,0.090909,0.045455,0.000000,0.045455,0.136364,0.181818,0.045455,0.000000,0.000000
8019,P32349,Training,5,Negative,MDELLGEALSAENQTGESTVESEKLVTPEDVMTISSLEQRTLNPDL...,0.090909,0.000000,0.045455,0.227273,0.000000,...,0.045455,0.045455,0.000000,0.045455,0.000000,0.136364,0.090909,0.045455,0.000000,0.000000
8020,G5EGE9,Training,5,Negative,MDVPSSSNVTGRRKRQVLDDDEDDGFRSTPLRKVRGTKKIRPADVV...,0.000000,0.000000,0.181818,0.045455,0.000000,...,0.045455,0.045455,0.045455,0.045455,0.136364,0.136364,0.045455,0.136364,0.000000,0.000000


#### Feature extraction

In [ ]:
#Hydrophobicity feature
window_size = 5
kd_scale = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I': 4.5,
    'L': 3.8, 'K': -3.9, 'M': 1.9, 'F': 2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V': 4.2}
eis_scale = {
    "A": 0.6, "R": -2.5, "N": -0.8, "D": -0.9, "C": 0.3,
    "Q": -0.9, "E": -0.7, "G": 0.5, "H": -0.4, "I": 1.4,
    "L": 1.1, "K": -1.5, "M": 0.6, "F": 1.2, "P": 0.1,
    "S": -0.2, "T": -0.1, "W": 0.8, "Y": 0.3, "V": 1.1
}
TM_scale = {
    "A": 0.4, "R": -2.6, "N": -1.6, "D": -3.3, "C": -0.3,
    "Q": -1.8, "E": -2.9, "G": -0.2, "H": -1.4, "I": 2.0,
    "L": 1.8, "K": -3.5, "M": 1.4, "F": 2.0, "P": -1.4,
    "S": -0.5, "T": -0.3, "W": 1.5, "Y": 0.5, "V": 1.5
}


for index,row in df.iterrows():
  seq = row["sequence"][:40]
  padding_size = int(window_size/2)
  protein_analysis = ProteinAnalysis("X" * padding_size + seq + "X"*padding_size)
  KD_profile = protein_analysis.protein_scale(kd_scale, window_size)
  eis_profile = protein_analysis.protein_scale(eis_scale, window_size)
  TM_profile = protein_analysis.protein_scale(TM_scale, window_size)
  #Kite Doolittle
  df.at[index, "KD average"] = np.mean(KD_profile)
  df.at[index, "KD maximal"] = max(KD_profile)
  df.at[index, "KD std"] = np.std(KD_profile)
  #Eisemberg
  df.at[index, "Eis average"] = np.mean(eis_profile)
  df.at[index, "Eis maximal"] = max(eis_profile)
  df.at[index, "Eis std"] = np.std(eis_profile)
  #Transmembrane tendency
  df.at[index, "TM average"] = np.mean(TM_profile)
  df.at[index, "TM maximal"] = max(TM_profile)
  df.at[index, "TM std"] = np.std(TM_profile)
df

Output streaming troncato alle ultime 5000 righe.


,protein_id,dataset_class,cv_subset,type,sequence,A_composition,C_composition,D_composition,E_composition,F_composition,...,Y_composition,KD average,KD maximal,KD std,Eis average,Eis maximal,Eis std,TM average,TM maximal,TM std
0,Q99MA2,Training,1,Positive,MAQAYWQCYPWLVLLCACAWSYPGPESLGREDVRDCSTNPPRLPVT...,0.181818,0.136364,0.000000,0.000000,0.000000,...,0.136364,-0.1625,3.62,1.508411,0.0980,1.04,0.483814,-0.2920,1.68,0.893597
1,P17948,Training,1,Positive,MVSYWDTGVLLCALLSCLLLTGSSSGSKLKDPELSLKGTQHIMQAG...,0.045455,0.090909,0.045455,0.000000,0.000000,...,0.045455,0.4205,3.22,1.622193,0.2125,0.84,0.415637,-0.0945,1.10,0.901887
2,P41271,Training,1,Positive,MMLRVLVGAVLPAMLLAAPPPINKLALFPDKSAWCEAKNITQIVGH...,0.181818,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.7535,2.72,1.349847,0.3110,0.88,0.376562,0.0150,1.16,0.826205
3,Q8I948,Training,1,Positive,MAFRMKLVVCIVLLSTLAVMSSADVYKGGGGGRYGGGRYGGGGGYG...,0.090909,0.045455,0.000000,0.000000,0.045455,...,0.000000,0.5555,3.92,1.697130,0.2390,1.00,0.399173,0.0860,1.36,0.721279
4,Q92154,Training,1,Positive,MELLVLTVLLMGTGCISAPWAAWMPPKMAALSGTCVQLPCRFDYPE...,0.136364,0.045455,0.000000,0.045455,0.000000,...,0.000000,1.0625,3.06,1.032678,0.4450,0.88,0.252260,0.3265,1.32,0.561042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8017,Q01981,Training,5,Negative,MPQPGSSVDFSNLLNPQNNTAIPAEVSNATASATMASGASLLPPMV...,0.045455,0.000000,0.045455,0.000000,0.045455,...,0.000000,-0.1490,1.22,0.960437,0.1090,0.52,0.246290,-0.2795,0.70,0.469755
8018,Q9H3N8,Training,5,Negative,MPDTNSTINLSLSTRVTLAFFMSLVAFAIMLGNALVILAFVVDKNL...,0.045455,0.000000,0.045455,0.000000,0.090909,...,0.000000,1.2330,3.62,1.460483,0.4275,1.08,0.453535,0.5200,1.54,0.780935
8019,P32349,Training,5,Negative,MDELLGEALSAENQTGESTVESEKLVTPEDVMTISSLEQRTLNPDL...,0.090909,0.000000,0.045455,0.227273,0.000000,...,0.000000,-0.3970,1.82,1.091362,-0.0095,0.56,0.331149,-0.6135,0.82,0.619312
8020,G5EGE9,Training,5,Negative,MDVPSSSNVTGRRKRQVLDDDEDDGFRSTPLRKVRGTKKIRPADVV...,0.000000,0.000000,0.181818,0.045455,0.000000,...,0.000000,-1.2350,0.90,1.172482,-0.4540,0.28,0.491125,-1.1225,0.40,0.850484


#### Spliting into training, validation and testing subsets

In [ ]:
def cv_subsets(df):
  subsets = set()
  for _, row in df.iterrows():
    index = row["cv_subset"]
    subsets.add(index)
  combinations={}
  for index in subsets:
    validation = index
    training = list(subsets - {index})
    combinations[validation] = training
  return combinations

cv_indexes = []
combination = cv_subsets(df)


# Get the index of each protein ID in the DataFrame X
for key in combination.keys():
  test = []
  train = []
  validation = []
  for index, row in df.iterrows():
    if row["cv_subset"] == key:
      # Append the integer index instead of the protein ID
      test.append(index)
    elif row["cv_subset"] == combination[key][3]:
      validation.append(index)
    else:
      # Append the integer index instead of the protein ID
      train.append(index)
  cv_indexes.append((train, validation, test))

print(cv_indexes[4][2][0:10])
# the format of cv_indexes is [cv subsets, from 0 to 4][train val test, from 0 to 2]


[700, 701, 702, 703, 704, 705, 706, 707, 708, 709]


#### Columns of the DataFrame to omit during training

In [ ]:
cols_to_exclude = ['sequence', 'dataset_class', 'cv_subset', 'type', "protein_id"]

X = df.loc[:, ~df.columns.isin(cols_to_exclude)]
y = df['type'].map({'Positive': 1, 'Negative': 0}).values

#### Baseline

In [ ]:
# We'll use a simple StandardScaler + RBF SVM pipeline
def svm_pipeline(C, gamma):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=C, gamma=gamma, random_state=42))
    ])

# Minimal manual grid
C_grid = [0.1, 1.0, 10.0, 100.0]
gamma_grid = ["scale", 0.01, 0.1, 1.0]
c = 0
for i in cv_indexes: #loop across cv folds
  best_mcc = -np.inf
  c += 1
  X_train = X.loc[i[0]]
  y_train = y[i[0]]
  X_val = X.loc[i[1]]
  y_val = y[i[1]]
  X_test = X.loc[i[2]]
  y_test = y[i[2]]
  for C in C_grid:
      for gamma in gamma_grid:
          pipe = svm_pipeline(C, gamma)
          pipe.fit(X_train, y_train)
          y_pred_val = pipe.predict(X_val)
          mcc_val = matthews_corrcoef(y_val, y_pred_val)
          if mcc_val > best_mcc:
                best_mcc = mcc_val
                actual_gamma = pipe.named_steps['svm']._gamma
                best_c_val = C
                best_gamma_val = actual_gamma
  #metrics
  pipe = svm_pipeline(best_c_val, best_gamma_val)
  pipe.fit(X_train, y_train)
  y_pred_test = pipe.predict(X_test)
  acc = accuracy_score(y_test, y_pred_test)
  precision = precision_score(y_test, y_pred_test, zero_division=0)
  recall = recall_score(y_test, y_pred_test, zero_division=0)
  f1 = f1_score(y_test, y_pred_test, zero_division=0)
  mcc = matthews_corrcoef(y_test, y_pred_test)
  print(f"Cross validation cycle: {c}")
  print(f"Best validation MCC: {best_mcc:.3f} with params C: {best_c_val:.2f} - gamma: {best_gamma_val:.4f}")
  print(f"Accuracy: {acc:.3f} - Precision: {precision:.3f} - Recall: {recall:.3f} - F1: {f1:.3f} - MCC: {mcc:.3f} \n")

Cross validation cycle: 1
Best validation MCC: 0.832 with params C: 10.00 - gamma: 0.0100
Accuracy: 0.966 - Precision: 0.838 - Recall: 0.857 - F1: 0.847 - MCC: 0.829 

Cross validation cycle: 2
Best validation MCC: 0.836 with params C: 1.00 - gamma: 0.0345
Accuracy: 0.964 - Precision: 0.831 - Recall: 0.846 - F1: 0.839 - MCC: 0.819 

Cross validation cycle: 3
Best validation MCC: 0.823 with params C: 10.00 - gamma: 0.0100
Accuracy: 0.967 - Precision: 0.832 - Recall: 0.874 - F1: 0.852 - MCC: 0.834 

Cross validation cycle: 4
Best validation MCC: 0.838 with params C: 10.00 - gamma: 0.0100
Accuracy: 0.969 - Precision: 0.857 - Recall: 0.857 - F1: 0.857 - MCC: 0.840 

Cross validation cycle: 5
Best validation MCC: 0.840 with params C: 1.00 - gamma: 0.0345
Accuracy: 0.966 - Precision: 0.854 - Recall: 0.834 - F1: 0.844 - MCC: 0.825 



#### Execution with feature selection using ElasticNet

In [ ]:
# We'll use a simple StandardScaler + RBF SVM pipeline
def svm_pipeline(C, gamma):
    return Pipeline([
        ("svm", SVC(kernel="rbf", C=C, gamma=gamma, random_state=42))
    ])

# Minimal manual grid
C_grid = [0.1, 1.0, 10.0, 100.0]
gamma_grid = ["scale", 0.01, 0.1, 1.0]

metrics_tot_en = []
features_tot_en = []
c = 0

for i in cv_indexes: #loop across cv folds
  c += 1
  print(f"Cross validation fold {c}")
  X_train = X.loc[i[0]]
  y_train = y[i[0]]
  X_val = X.loc[i[1]]
  y_val = y[i[1]]
  X_test = X.loc[i[2]]
  y_test = y[i[2]]
  best_mcc = -np.inf
  #scaling
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_val_scaled = scaler.transform(X_val)
  X_test_scaled = scaler.transform(X_test)
  #feature selection
  model_feature_sel = LogisticRegressionCV(
    penalty='elasticnet',
    l1_ratios=[0.1, 0.5, 0.7, 0.9, 1.0],
    cv=3,
    solver='saga',
    random_state=42,
    max_iter=5000,
  )
  selector = SelectFromModel(model_feature_sel, threshold="median")
  selector.fit(X_train_scaled, y_train)
  selected_mask = selector.get_support()
  selected_features = X_train.columns[selected_mask].tolist()
  print(f"Selected features by ElasticNet: {selected_features}")
  features_tot_en.append(selected_features)
  #scaling
  X_train_selected = selector.transform(X_train_scaled)
  X_val_selected = selector.transform(X_val_scaled)
  X_test_selected = selector.transform(X_test_scaled)
  #grid search
  for C in C_grid:
      for gamma in gamma_grid:
          pipe = svm_pipeline(C, gamma)
          pipe.fit(X_train_selected, y_train)
          y_pred_val = pipe.predict(X_val_selected)

          mcc_val = matthews_corrcoef(y_val, y_pred_val)
          if mcc_val > best_mcc:
                best_mcc = mcc_val
                actual_gamma = pipe.named_steps['svm']._gamma
                best_c_val = C
                best_gamma_val = actual_gamma
  #metrics
  pipe = svm_pipeline(best_c_val, best_gamma_val)
  pipe.fit(X_train_selected, y_train)
  y_pred_test = pipe.predict(X_test_selected)
  acc = accuracy_score(y_test, y_pred_test)
  precision = precision_score(y_test, y_pred_test, zero_division=0)
  recall = recall_score(y_test, y_pred_test, zero_division=0)
  f1 = f1_score(y_test, y_pred_test, zero_division=0)
  mcc = matthews_corrcoef(y_test, y_pred_test)
  metrics_tot_en.append([precision, recall, f1, acc, mcc, best_c_val, best_gamma_val])
  print(f"Best validation MCC: {best_mcc:.3f} with params C: {best_c_val:.2f} - gamma: {best_gamma_val:.4f}")
  print(f"Accuracy: {acc:.3f} - Precision: {precision:.3f} - Recall: {recall:.3f} - F1: {f1:.3f} - MCC: {mcc:.3f} \n")

Cross validation fold 1
Selected features by ElasticNet: ['A_composition', 'D_composition', 'E_composition', 'K_composition', 'L_composition', 'N_composition', 'P_composition', 'W_composition', 'KD average', 'KD std', 'Eis average', 'Eis maximal', 'Eis std', 'TM average', 'TM maximal']
Best validation MCC: 0.839 with params C: 10.00 - gamma: 0.1000
Accuracy: 0.958 - Precision: 0.813 - Recall: 0.794 - F1: 0.803 - MCC: 0.780 

Cross validation fold 2
Selected features by ElasticNet: ['A_composition', 'D_composition', 'E_composition', 'G_composition', 'K_composition', 'L_composition', 'N_composition', 'W_composition', 'KD maximal', 'KD std', 'Eis average', 'Eis maximal', 'Eis std', 'TM average', 'TM maximal']
Best validation MCC: 0.823 with params C: 10.00 - gamma: 0.0667
Accuracy: 0.962 - Precision: 0.817 - Recall: 0.840 - F1: 0.828 - MCC: 0.807 

Cross validation fold 3
Selected features by ElasticNet: ['A_composition', 'D_composition', 'E_composition', 'K_composition', 'L_composition',

#### Printing ElasticNet performances

In [ ]:
precision=[]
recall=[]
f1_scores_list=[]
accuracy=[]
mcc=[]
c=[]
gamma=[]
features=[]

for i in metrics_tot_en:
  precision.append(i[0])
  recall.append(i[1])
  f1_scores_list.append(i[2])
  accuracy.append(i[3])
  mcc.append(i[4])
  c.append(i[5])
  gamma.append(i[6])

metrics_df = pd.DataFrame({"Precision": precision, "Recall": recall, "F1 score": f1_scores_list, "Accuracy": accuracy, "MCC": mcc, "C":c, 'Gamma':gamma})
print(metrics_df.head())
print()
metrics_description = metrics_df.describe()
plus_minus = "\u00b1"
print("Precision: ",round(metrics_description["Precision"]["mean"],4),plus_minus,round(metrics_description["Precision"]["std"],4))
print("Recall: ",round(metrics_description["Recall"]["mean"],4),plus_minus,round(metrics_description["Recall"]["std"],4))
print("F1 score: ",round(metrics_description["F1 score"]["mean"],4),plus_minus,round(metrics_description["F1 score"]["std"],4))
print("Accuracy: ",round(metrics_description["Accuracy"]["mean"],4),plus_minus,round(metrics_description["Accuracy"]["std"],4))
print("MCC: ",round(metrics_description["MCC"]["mean"],4),plus_minus,round(metrics_description["MCC"]["std"],4))

   Precision    Recall  F1 score  Accuracy       MCC      C     Gamma
0   0.812865  0.794286  0.803468  0.957606  0.779778   10.0  0.100000
1   0.816667  0.840000  0.828169  0.961970  0.806894   10.0  0.066667
2   0.817680  0.845714  0.831461  0.962594  0.810575  100.0  0.010000
3   0.847953  0.828571  0.838150  0.965087  0.818655   10.0  0.066667
4   0.804348  0.845714  0.824513  0.960772  0.802766   10.0  0.100000

Precision:  0.8199 ± 0.0165
Recall:  0.8309 ± 0.0216
F1 score:  0.8252 ± 0.0131
Accuracy:  0.9616 ± 0.0027
MCC:  0.8037 ± 0.0146


#### Execution with feature selection using RandomForest & SelectFromModel

In [ ]:
# We'll use a simple StandardScaler + RBF SVM pipeline
def svm_pipeline(C, gamma):
    return Pipeline([
        ("svm", SVC(kernel="rbf", C=C, gamma=gamma, random_state=42))
    ])

# Minimal manual grid
C_grid = [0.1, 1.0, 10.0, 100.0]
gamma_grid = ["scale", 0.01, 0.1, 1.0]

metrics_tot_rf = []
features_tot_rf = []
c = 0

for i in cv_indexes: #loop across cv folds
  c += 1
  print(f"Cross validation fold {c}")
  X_train = X.loc[i[0]]
  y_train = y[i[0]]
  X_val = X.loc[i[1]]
  y_val = y[i[1]]
  X_test = X.loc[i[2]]
  y_test = y[i[2]]
  best_mcc = -np.inf
  #scaling
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_val_scaled = scaler.transform(X_val)
  X_test_scaled = scaler.transform(X_test)
  #feature selection
  model_feature_sel = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
  )
  selector = SelectFromModel(model_feature_sel, threshold="median")
  selector.fit(X_train_scaled, y_train)
  selected_mask = selector.get_support()
  selected_features = X_train.columns[selected_mask].tolist()
  print(f"Selected features by RandomForest: {selected_features}")
  features_tot_rf.append(selected_features)
  #scaling
  X_train_selected = selector.transform(X_train_scaled)
  X_val_selected = selector.transform(X_val_scaled)
  X_test_selected = selector.transform(X_test_scaled)
  #grid search
  for C in C_grid:
      for gamma in gamma_grid:
          pipe = svm_pipeline(C, gamma)
          pipe.fit(X_train_selected, y_train)
          y_pred_val = pipe.predict(X_val_selected)

          mcc_val = matthews_corrcoef(y_val, y_pred_val)
          if mcc_val > best_mcc:
                best_mcc = mcc_val
                actual_gamma = pipe.named_steps['svm']._gamma
                best_c_val = C
                best_gamma_val = actual_gamma
  #metrics
  pipe = svm_pipeline(best_c_val, best_gamma_val)
  pipe.fit(X_train_selected, y_train)
  y_pred_test = pipe.predict(X_test_selected)
  acc = accuracy_score(y_test, y_pred_test)
  precision = precision_score(y_test, y_pred_test, zero_division=0)
  recall = recall_score(y_test, y_pred_test, zero_division=0)
  f1 = f1_score(y_test, y_pred_test, zero_division=0)
  mcc = matthews_corrcoef(y_test, y_pred_test)
  metrics_tot_rf.append([precision, recall, f1, acc, mcc, best_c_val, best_gamma_val])
  print(f"Best validation MCC: {best_mcc:.3f} with params C: {best_c_val:.2f} - gamma: {best_gamma_val:.4f}")
  print(f"Accuracy: {acc:.3f} - Precision: {precision:.3f} - Recall: {recall:.3f} - F1: {f1:.3f} - MCC: {mcc:.3f} \n")

Cross validation fold 1
Selected features by RandomForest: ['A_composition', 'D_composition', 'E_composition', 'L_composition', 'N_composition', 'R_composition', 'KD average', 'KD maximal', 'KD std', 'Eis average', 'Eis maximal', 'Eis std', 'TM average', 'TM maximal', 'TM std']
Best validation MCC: 0.806 with params C: 100.00 - gamma: 0.0100
Accuracy: 0.959 - Precision: 0.815 - Recall: 0.806 - F1: 0.810 - MCC: 0.787 

Cross validation fold 2
Selected features by RandomForest: ['A_composition', 'D_composition', 'E_composition', 'G_composition', 'L_composition', 'N_composition', 'KD average', 'KD maximal', 'KD std', 'Eis average', 'Eis maximal', 'Eis std', 'TM average', 'TM maximal', 'TM std']
Best validation MCC: 0.830 with params C: 100.00 - gamma: 0.0100
Accuracy: 0.961 - Precision: 0.819 - Recall: 0.829 - F1: 0.824 - MCC: 0.802 

Cross validation fold 3
Selected features by RandomForest: ['A_composition', 'D_composition', 'E_composition', 'K_composition', 'L_composition', 'N_composit

#### Printing RandomForest-SelectFromModel performances

In [ ]:
precision=[]
recall=[]
f1_scores_list=[]
accuracy=[]
mcc=[]
c=[]
gamma=[]
features=[]

for i in metrics_tot_rf:
  precision.append(i[0])
  recall.append(i[1])
  f1_scores_list.append(i[2])
  accuracy.append(i[3])
  mcc.append(i[4])
  c.append(i[5])
  gamma.append(i[6])

metrics_df = pd.DataFrame({"Precision": precision, "Recall": recall, "F1 score": f1_scores_list, "Accuracy": accuracy, "MCC": mcc, "C":c, 'Gamma':gamma})
print(metrics_df.head())
print()
metrics_description = metrics_df.describe()
plus_minus = "\u00b1"
print("Precision: ",round(metrics_description["Precision"]["mean"],4),plus_minus,round(metrics_description["Precision"]["std"],4))
print("Recall: ",round(metrics_description["Recall"]["mean"],4),plus_minus,round(metrics_description["Recall"]["std"],4))
print("F1 score: ",round(metrics_description["F1 score"]["mean"],4),plus_minus,round(metrics_description["F1 score"]["std"],4))
print("Accuracy: ",round(metrics_description["Accuracy"]["mean"],4),plus_minus,round(metrics_description["Accuracy"]["std"],4))
print("MCC: ",round(metrics_description["MCC"]["mean"],4),plus_minus,round(metrics_description["MCC"]["std"],4))

   Precision    Recall  F1 score  Accuracy       MCC      C     Gamma
0   0.815029  0.805714  0.810345  0.958853  0.787285  100.0  0.010000
1   0.819209  0.828571  0.823864  0.961347  0.802172  100.0  0.010000
2   0.830508  0.840000  0.835227  0.963840  0.814936   10.0  0.100000
3   0.831395  0.817143  0.824207  0.961970  0.802926  100.0  0.010000
4   0.828571  0.828571  0.828571  0.962640  0.807607   10.0  0.066667

Precision:  0.8249 ± 0.0074
Recall:  0.824 ± 0.013
F1 score:  0.8244 ± 0.0091
Accuracy:  0.9617 ± 0.0019
MCC:  0.803 ± 0.0101


#### Execution with feature selection using RandomForest & Gini importances

In [ ]:
# We'll use a simple StandardScaler + RBF SVM pipeline
def svm_pipeline(C, gamma):
    return Pipeline([
        ("svm", SVC(kernel="rbf", C=C, gamma=gamma, random_state=42))
    ])

# Minimal manual grid
C_grid = [0.1, 1.0, 10.0, 100.0]
gamma_grid = ["scale", 0.01, 0.1, 1.0]

metrics_tot_rf2 = []
features_tot_rf2 = []
c = 0
feature_names = np.array([f"f{i:02d}" for i in range(X.shape[1])])

for i in cv_indexes: #loop across cv folds
  c += 1
  print(f"Cross validation fold {c}")
  X_train = X.loc[i[0]].values # Convert to NumPy array
  y_train = y[i[0]]
  X_val = X.loc[i[1]].values # Convert to NumPy array
  y_val = y[i[1]]
  X_test = X.loc[i[2]].values # Convert to NumPy array
  y_test = y[i[2]]
  best_mcc = -np.inf
  #scaling
  scaler = StandardScaler()
  X_train_scaled = scaler.fit_transform(X_train)
  X_val_scaled = scaler.transform(X_val)
  X_test_scaled = scaler.transform(X_test)
  #feature selection
  rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
  )
  rf.fit(X_train, y_train)  # fit only on TRAIN

  gini_imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
  gini_df = gini_imp.reset_index()
  gini_df.columns = ["feature", "importance"]

  def accuracy_on_subset(C, gamma, subset_features):
    # subset by feature names
    idx = [np.where(feature_names == f)[0][0] for f in subset_features]
    Xtr = X_train[:, idx]
    Xva = X_val[:, idx]
    pipe = svm_pipeline(C, gamma)
    pipe.fit(Xtr, y_train)     # train on TRAIN only
    return pipe.score(Xva, y_val)  # accuracy on VALIDATION

  # We'll sweep k and, for each k, re-evaluate the best baseline SVM params on the reduced feature set
  ks = list(range(2, min(26, X_train.shape[1]+1)))  # keep it small for speed/clarity
  curve = []

  for k in ks:
      subset = gini_df["feature"].head(k).tolist()
      acc_k = accuracy_on_subset(10, 0.01, subset)
      curve.append(acc_k)

  best_k_idx = int(np.argmax(curve))
  best_k = ks[best_k_idx]

  # Use the best k from the validation curve
  best_subset = gini_df["feature"].head(best_k).tolist()
  idx = [np.where(feature_names == f)[0][0] for f in best_subset]

  Xtr_sel = X_train[:, idx]
  Xva_sel = X_val[:, idx]
  Xte_sel = X_test[:, idx]

  # Manual grid search again but now restricted to the selected features
  best_score_sel = -np.inf
  best_params_sel = None

  #grid search
  for C in C_grid:
      for gamma in gamma_grid:
          pipe = svm_pipeline(C, gamma)
          pipe.fit(Xtr_sel, y_train)
          y_pred_val = pipe.predict(Xva_sel)

          mcc_val = matthews_corrcoef(y_val, y_pred_val)
          if mcc_val > best_mcc:
                best_mcc = mcc_val
                actual_gamma = pipe.named_steps['svm']._gamma
                best_c_val = C
                best_gamma_val = actual_gamma
  #metrics
  pipe = svm_pipeline(best_c_val, best_gamma_val)
  pipe.fit(Xtr_sel, y_train)
  y_pred_test = pipe.predict(Xte_sel)
  acc = accuracy_score(y_test, y_pred_test)
  precision = precision_score(y_test, y_pred_test, zero_division=0)
  recall = recall_score(y_test, y_pred_test, zero_division=0)
  f1 = f1_score(y_test, y_pred_test, zero_division=0)
  mcc = matthews_corrcoef(y_test, y_pred_test)
  metrics_tot_rf2.append([precision, recall, f1, acc, mcc, best_c_val, best_gamma_val])
  print(f"Best validation MCC: {best_mcc:.3f} with params C: {best_c_val:.2f} - gamma: {best_gamma_val:.4f}")
  print(f"Accuracy: {acc:.3f} - Precision: {precision:.3f} - Recall: {recall:.3f} - F1: {f1:.3f} - MCC: {mcc:.3f} \n")

Cross validation fold 1
Best validation MCC: 0.813 with params C: 100.00 - gamma: 0.1271
Accuracy: 0.961 - Precision: 0.826 - Recall: 0.811 - F1: 0.818 - MCC: 0.796 

Cross validation fold 2
Best validation MCC: 0.819 with params C: 100.00 - gamma: 0.1000
Accuracy: 0.967 - Precision: 0.855 - Recall: 0.840 - F1: 0.847 - MCC: 0.829 

Cross validation fold 3
Best validation MCC: 0.769 with params C: 100.00 - gamma: 0.1000
Accuracy: 0.953 - Precision: 0.789 - Recall: 0.771 - F1: 0.780 - MCC: 0.754 

Cross validation fold 4
Best validation MCC: 0.816 with params C: 100.00 - gamma: 0.1000
Accuracy: 0.965 - Precision: 0.848 - Recall: 0.829 - F1: 0.838 - MCC: 0.819 

Cross validation fold 5
Best validation MCC: 0.851 with params C: 100.00 - gamma: 0.1000
Accuracy: 0.965 - Precision: 0.851 - Recall: 0.817 - F1: 0.834 - MCC: 0.814 



#### Printing RandomForest-Gini performances

In [ ]:
precision=[]
recall=[]
f1_scores_list=[]
accuracy=[]
mcc=[]
c=[]
gamma=[]
features=[]

for i in metrics_tot_rf2:
  precision.append(i[0])
  recall.append(i[1])
  f1_scores_list.append(i[2])
  accuracy.append(i[3])
  mcc.append(i[4])
  c.append(i[5])
  gamma.append(i[6])

metrics_df = pd.DataFrame({"Precision": precision, "Recall": recall, "F1 score": f1_scores_list, "Accuracy": accuracy, "MCC": mcc, "C":c, 'Gamma':gamma})
print(metrics_df.head())
print()
metrics_description = metrics_df.describe()
plus_minus = "\u00b1"
print("Precision: ",round(metrics_description["Precision"]["mean"],4),plus_minus,round(metrics_description["Precision"]["std"],4))
print("Recall: ",round(metrics_description["Recall"]["mean"],4),plus_minus,round(metrics_description["Recall"]["std"],4))
print("F1 score: ",round(metrics_description["F1 score"]["mean"],4),plus_minus,round(metrics_description["F1 score"]["std"],4))
print("Accuracy: ",round(metrics_description["Accuracy"]["mean"],4),plus_minus,round(metrics_description["Accuracy"]["std"],4))
print("MCC: ",round(metrics_description["MCC"]["mean"],4),plus_minus,round(metrics_description["MCC"]["std"],4))

   Precision    Recall  F1 score  Accuracy       MCC      C     Gamma
0   0.825581  0.811429  0.818444  0.960723  0.796463  100.0  0.127144
1   0.854651  0.840000  0.847262  0.966958  0.828778  100.0  0.100000
2   0.789474  0.771429  0.780347  0.952618  0.753859  100.0  0.100000
3   0.847953  0.828571  0.838150  0.965087  0.818655  100.0  0.100000
4   0.851190  0.817143  0.833819  0.964508  0.814173  100.0  0.100000

Precision:  0.8338 ± 0.0272
Recall:  0.8137 ± 0.0261
F1 score:  0.8236 ± 0.0263
Accuracy:  0.962 ± 0.0057
MCC:  0.8024 ± 0.0295
